In [ ]:
# Mount Google Drive and load the shared project configuration

from google.colab import drive
from pathlib import Path

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/NLP04_Toxic_Comment_Classification')

CONFIG_DIR = PROJECT_ROOT / '00_config'
RAW_DIR = PROJECT_ROOT / '01_data' / 'raw'

NOTEBOOK_DIR = PROJECT_ROOT / '02_notebooks' / '01_dataset_eda'
RESULTS_DIR = NOTEBOOK_DIR / 'results'

FIGURES_DIR = RESULTS_DIR / 'figures'
TABLES_DIR = RESULTS_DIR / 'tables'

assert NOTEBOOK_DIR.is_dir(), "Notebook directory not found"

for folder in [RESULTS_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Load project configuration
with open(CONFIG_DIR / 'project_config.json', 'r') as file:
    config = json.load(file)

LABELS = config['labels']
SEED = config['seed']

print("Notebook:", NOTEBOOK_DIR.name)
print("Dataset directory:", RAW_DIR)
print("Figures directory:", FIGURES_DIR)
print("Tables directory:", TABLES_DIR)
print("Target labels:", LABELS)
print("Random seed:", SEED)

In [ ]:
# Verify dataset fingerprints and load the original CSV files

import hashlib

manifest_path = (
    PROJECT_ROOT / '02_notebooks' / '00_project_setup'
    / 'results' / 'tables' / 'dataset_manifest.csv'
)

manifest_df = pd.read_csv(manifest_path)

# Verify that raw datasets remain unchanged
for _, row in manifest_df.iterrows():
    path = RAW_DIR / row['filename']
    sha256 = hashlib.sha256()

    with open(path, 'rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            sha256.update(chunk)

    assert sha256.hexdigest() == row['sha256'], \
        f"Dataset integrity check failed: {path.name}"

    print(f"{path.name}: Integrity verified")

# Load datasets
train_df = pd.read_csv(RAW_DIR / 'train.csv')
test_df = pd.read_csv(RAW_DIR / 'test.csv')
test_labels_df = pd.read_csv(RAW_DIR / 'test_labels.csv')

print("\nDataset dimensions")
print("Training data:", train_df.shape)
print("Test data:", test_df.shape)
print("Test labels:", test_labels_df.shape)

In [ ]:
# Summarize dataset dimensions and available columns

datasets = {
    'Training data': train_df,
    'Test data': test_df,
    'Test labels': test_labels_df
}

overview = []

for name, df in datasets.items():
    overview.append({
        'Dataset': name,
        'Rows': df.shape[0],
        'Columns': df.shape[1],
        'Unique IDs': df['id'].nunique(),
        'Text column': 'comment_text' in df.columns,
        'Label columns': sum(label in df.columns for label in LABELS)
    })

overview_df = pd.DataFrame(overview)

display(overview_df)

# Display column names and data types
print("\nTraining dataset structure:")

schema_df = pd.DataFrame({
    'Column': train_df.columns,
    'Data type': train_df.dtypes.astype(str).values
})

display(schema_df)

# Save dataset overview and schema
overview_df.to_csv(TABLES_DIR / 'dataset_overview.csv', index=False)
schema_df.to_csv(TABLES_DIR / 'training_schema.csv', index=False)

In [ ]:
# Check missing values, empty comments, duplicate IDs and invalid labels

quality_records = []

for name, df in datasets.items():

    empty_comments = 0

    if 'comment_text' in df.columns:
        empty_comments = df['comment_text'].fillna('').str.strip().eq('').sum()

    quality_records.append({
        'Dataset': name,
        'Missing values': df.isnull().sum().sum(),
        'Missing IDs': df['id'].isnull().sum(),
        'Duplicate IDs': df['id'].duplicated().sum(),
        'Empty comments': empty_comments
    })

quality_df = pd.DataFrame(quality_records)

print("Data quality summary")
display(quality_df)

# Check valid label values
label_validity = []

for label in LABELS:
    label_validity.append({
        'Label': label,
        'Training values': sorted(train_df[label].unique().tolist()),
        'Test values': sorted(test_labels_df[label].unique().tolist()),
        'Invalid training labels': (~train_df[label].isin([0, 1])).sum(),
        'Invalid test labels': (~test_labels_df[label].isin([-1, 0, 1])).sum()
    })

validity_df = pd.DataFrame(label_validity)

print("\nLabel validity")
display(validity_df)

# Verify correspondence between test comments and labels
matching_ids = set(test_df['id']) == set(test_labels_df['id'])

print("\nTest and test label IDs match:", matching_ids)

# Save data quality tables
quality_df.to_csv(TABLES_DIR / 'data_quality_summary.csv', index=False)
validity_df.to_csv(TABLES_DIR / 'label_validity.csv', index=False)

In [ ]:
# Examine test label availability and identify evaluable comments

test_merged = test_df.merge(
    test_labels_df,
    on='id',
    how='inner',
    validate='one_to_one'
)

# Identify fully labelled and unlabelled test comments
valid_mask = test_merged[LABELS].isin([0, 1]).all(axis=1)
unlabelled_mask = test_merged[LABELS].eq(-1).all(axis=1)
partial_mask = ~(valid_mask | unlabelled_mask)

test_availability_df = pd.DataFrame({
    'Category': [
        'Fully labelled',
        'Unlabelled',
        'Partially labelled'
    ],
    'Comments': [
        valid_mask.sum(),
        unlabelled_mask.sum(),
        partial_mask.sum()
    ]
})

test_availability_df['Percentage'] = (
    test_availability_df['Comments'] / len(test_merged) * 100
).round(2)

display(test_availability_df)

# Identify the official evaluable test records
test_valid_df = test_merged.loc[valid_mask].copy()

print("Total test comments:", len(test_merged))
print("Evaluable test comments:", len(test_valid_df))

# Inspect empty test comments
empty_mask = test_merged['comment_text'].str.strip().eq('')

print("\nEmpty test comment details:")
display(test_merged.loc[empty_mask, ['id'] + LABELS])

print("Empty comments in evaluable test set:",
      (empty_mask & valid_mask).sum())

# Save test availability statistics
test_availability_df.to_csv(
    TABLES_DIR / 'test_label_availability.csv',
    index=False
)

test_merged.loc[empty_mask, ['id'] + LABELS].to_csv(
    TABLES_DIR / 'empty_test_comments.csv',
    index=False
)

In [ ]:
# Analyse the distribution of clean and toxic training comments

label_counts = train_df[LABELS].sum(axis=1)

clean_count = (label_counts == 0).sum()
toxic_count = (label_counts > 0).sum()

distribution_df = pd.DataFrame({
    'Category': ['Clean', 'Toxic'],
    'Count': [clean_count, toxic_count]
})

distribution_df['Percentage'] = (
    distribution_df['Count'] / len(train_df) * 100
).round(2)

display(distribution_df)

# Save distribution statistics
distribution_df.to_csv(
    TABLES_DIR / 'clean_toxic_distribution.csv',
    index=False
)

# Visualize the distribution
sns.set_theme(style='whitegrid')

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    distribution_df['Category'],
    distribution_df['Count'],
    color=['#4C78A8', '#E45756'],
    width=0.5
)

for bar, count, percentage in zip(
    bars,
    distribution_df['Count'],
    distribution_df['Percentage']
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 2000,
        f'{count:,}\n({percentage:.2f}%)',
        ha='center',
        fontsize=11
    )

ax.set_title('Distribution of Clean and Toxic Comments', fontsize=14)
ax.set_xlabel('Comment Category', fontsize=12)
ax.set_ylabel('Number of Comments', fontsize=12)
ax.set_ylim(0, distribution_df['Count'].max() * 1.18)
ax.ticklabel_format(style='plain', axis='y')

sns.despine()
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / 'clean_toxic_distribution.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
# Analyse individual label frequencies and class imbalance

label_stats = pd.DataFrame({
    'Label': LABELS,
    'Positive': train_df[LABELS].sum().values,
    'Negative': (len(train_df) - train_df[LABELS].sum()).values
})

label_stats['Positive (%)'] = (
    label_stats['Positive'] / len(train_df) * 100
).round(3)

label_stats['Negative (%)'] = (
    label_stats['Negative'] / len(train_df) * 100
).round(3)

label_stats['Imbalance Ratio'] = (
    label_stats['Negative'] / label_stats['Positive']
).round(2)

display(label_stats)

label_stats.to_csv(
    TABLES_DIR / 'label_statistics.csv',
    index=False
)

# Consistent label colours for subsequent visualizations
LABEL_COLORS = {
    'toxic': '#4C78A8',
    'severe_toxic': '#F58518',
    'obscene': '#54A24B',
    'threat': '#E45756',
    'insult': '#B279A2',
    'identity_hate': '#72B7B2'
}

# Plot positive label frequencies
plot_df = label_stats.sort_values('Positive', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(
    plot_df['Label'],
    plot_df['Positive'],
    color=[LABEL_COLORS[label] for label in plot_df['Label']],
    width=0.65
)

for bar, count, percentage in zip(
    bars, plot_df['Positive'], plot_df['Positive (%)']
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 200,
        f'{count:,}\n({percentage:.2f}%)',
        ha='center',
        fontsize=10
    )

ax.set_title('Distribution of Toxicity Labels', fontsize=14)
ax.set_xlabel('Toxicity Label')
ax.set_ylabel('Number of Positive Comments')
ax.set_ylim(0, plot_df['Positive'].max() * 1.18)
ax.grid(axis='x', visible=False)

sns.despine()
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / 'label_distribution.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()
plt.close(fig)

# Plot negative-to-positive imbalance ratios
ratio_df = label_stats.sort_values(
    'Imbalance Ratio', ascending=True
)

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(
    ratio_df['Label'],
    ratio_df['Imbalance Ratio'],
    color=[LABEL_COLORS[label] for label in ratio_df['Label']]
)

for bar, ratio in zip(bars, ratio_df['Imbalance Ratio']):
    ax.text(
        ratio * 1.05,
        bar.get_y() + bar.get_height() / 2,
        f'{ratio:.1f}:1',
        va='center',
        fontsize=11
    )

ax.set_xscale('log')
ax.set_xlim(1, ratio_df['Imbalance Ratio'].max() * 2)
ax.set_title('Class Imbalance Across Toxicity Labels', fontsize=14)
ax.set_xlabel('Negative-to-Positive Ratio (Log Scale)')
ax.set_ylabel('Toxicity Label')
ax.grid(axis='y', visible=False)

sns.despine()
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / 'class_imbalance_ratio.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()
plt.close(fig)

In [ ]:
# Analyse the number of positive labels assigned to each comment

label_cardinality = train_df[LABELS].sum(axis=1)

cardinality_df = (
    label_cardinality.value_counts()
    .reindex(range(len(LABELS) + 1), fill_value=0)
    .sort_index()
    .rename_axis('Number of Labels')
    .reset_index(name='Count')
)

cardinality_df['Percentage of All Comments'] = (
    cardinality_df['Count'] / len(train_df) * 100
).round(3)

cardinality_df['Percentage of Toxic Comments'] = np.where(
    cardinality_df['Number of Labels'] > 0,
    cardinality_df['Count'] / toxic_count * 100,
    np.nan
).round(3)

display(cardinality_df)

# Calculate multi-label statistics
single_label_count = (label_cardinality == 1).sum()
multi_label_count = (label_cardinality > 1).sum()

cardinality_summary = pd.DataFrame({
    'Metric': [
        'Average labels per comment',
        'Average labels per toxic comment',
        'Label density',
        'Single-label comments',
        'Multi-label comments',
        'Multi-label percentage among toxic comments'
    ],
    'Value': [
        label_cardinality.mean(),
        label_cardinality[label_cardinality > 0].mean(),
        label_cardinality.mean() / len(LABELS),
        single_label_count,
        multi_label_count,
        multi_label_count / toxic_count * 100
    ]
})

display(cardinality_summary.round(4))

# Save statistics
cardinality_df.to_csv(
    TABLES_DIR / 'label_cardinality.csv', index=False
)

cardinality_summary.to_csv(
    TABLES_DIR / 'label_cardinality_summary.csv', index=False
)

# Plot label cardinality among toxic comments
plot_df = cardinality_df[
    cardinality_df['Number of Labels'] > 0
]

fig, ax = plt.subplots(figsize=(10, 5.5))

bars = ax.bar(
    plot_df['Number of Labels'],
    plot_df['Count'],
    color='#4C78A8',
    width=0.6
)

for bar, count, percentage in zip(
    bars,
    plot_df['Count'],
    plot_df['Percentage of Toxic Comments']
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 80,
        f'{count:,}\n({percentage:.2f}%)',
        ha='center',
        fontsize=10
    )

ax.set_title('Label Cardinality Among Toxic Comments', fontsize=14)
ax.set_xlabel('Number of Positive Toxicity Labels')
ax.set_ylabel('Number of Comments')
ax.set_xticks(range(1, len(LABELS) + 1))
ax.set_ylim(0, plot_df['Count'].max() * 1.20)
ax.grid(axis='x', visible=False)

sns.despine()
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / 'label_cardinality_distribution.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

In [ ]:
# Analyse relationships and co-occurrence between toxicity labels

label_matrix = train_df[LABELS].astype('int64')

# Absolute co-occurrence counts
cooccurrence = label_matrix.T.dot(label_matrix)

print("Label co-occurrence counts")
display(cooccurrence)

# Conditional co-occurrence percentages
conditional_cooccurrence = (
    cooccurrence.div(label_matrix.sum(), axis=0) * 100
)

print("\nConditional co-occurrence percentages")
display(conditional_cooccurrence.round(2))

# Save co-occurrence matrices
cooccurrence.to_csv(
    TABLES_DIR / 'label_cooccurrence_counts.csv'
)

conditional_cooccurrence.to_csv(
    TABLES_DIR / 'label_cooccurrence_percentages.csv'
)

# Absolute co-occurrence heatmap
fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(
    cooccurrence,
    annot=True,
    fmt=',d',
    cmap='Blues',
    linewidths=0.5,
    square=True,
    cbar_kws={'label': 'Number of Comments'},
    ax=ax
)

ax.set_title('Toxicity Label Co-occurrence Counts', fontsize=14)
ax.set_xlabel('Toxicity Label')
ax.set_ylabel('Toxicity Label')

plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / 'label_cooccurrence_counts.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

# Conditional co-occurrence heatmap
fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(
    conditional_cooccurrence,
    annot=True,
    fmt='.1f',
    cmap='YlGnBu',
    vmin=0,
    vmax=100,
    linewidths=0.5,
    square=True,
    cbar_kws={'label': 'Conditional Percentage (%)'},
    ax=ax
)

ax.set_title('Conditional Toxicity Label Co-occurrence', fontsize=14)
ax.set_xlabel('Co-occurring Label')
ax.set_ylabel('Given Label')

plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / 'label_cooccurrence_percentages.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

In [ ]:
# Calculate word and character lengths of training comments

word_lengths = train_df['comment_text'].str.split().str.len()
char_lengths = train_df['comment_text'].str.len()

length_stats = pd.DataFrame({
    'Statistic': [
        'Minimum', 'Mean', 'Median', 'Standard deviation',
        '75th percentile', '90th percentile', '95th percentile',
        '99th percentile', 'Maximum'
    ],
    'Word Length': [
        word_lengths.min(),
        word_lengths.mean(),
        word_lengths.median(),
        word_lengths.std(),
        word_lengths.quantile(0.75),
        word_lengths.quantile(0.90),
        word_lengths.quantile(0.95),
        word_lengths.quantile(0.99),
        word_lengths.max()
    ],
    'Character Length': [
        char_lengths.min(),
        char_lengths.mean(),
        char_lengths.median(),
        char_lengths.std(),
        char_lengths.quantile(0.75),
        char_lengths.quantile(0.90),
        char_lengths.quantile(0.95),
        char_lengths.quantile(0.99),
        char_lengths.max()
    ]
})

display(length_stats.round(2))

length_stats.to_csv(
    TABLES_DIR / 'comment_length_statistics.csv',
    index=False
)

# Plot word and character length distributions
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

for ax, lengths, name, color in zip(
    axes,
    [word_lengths, char_lengths],
    ['Word', 'Character'],
    ['#4C78A8', '#54A24B']
):
    upper_limit = lengths.quantile(0.99)

    sns.histplot(
        lengths,
        bins=60,
        binrange=(0, upper_limit),
        color=color,
        ax=ax
    )

    ax.axvline(
        lengths.median(),
        color='#E45756',
        linestyle='--',
        linewidth=2,
        label=f'Median: {lengths.median():.0f}'
    )

    ax.axvline(
        lengths.quantile(0.95),
        color='#F58518',
        linestyle='--',
        linewidth=2,
        label=f'95th percentile: {lengths.quantile(0.95):.0f}'
    )

    ax.set_title(f'{name} Length Distribution')
    ax.set_xlabel(f'Number of {name}s')
    ax.set_ylabel('Number of Comments')
    ax.set_xlim(0, upper_limit)
    ax.legend()
    ax.grid(axis='x', visible=False)

fig.suptitle(
    'Training Comment Length Distributions',
    fontsize=16
)

fig.text(
    0.5, 0.01,
    'Display limited to the 99th percentile for readability.',
    ha='center',
    fontsize=10
)

sns.despine()
plt.tight_layout(rect=[0, 0.04, 1, 0.95])

plt.savefig(
    FIGURES_DIR / 'comment_length_distributions.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

In [ ]:
# Compare comment lengths across clean and toxicity label groups

length_records = []
length_groups = {}

groups = {'clean': label_cardinality.eq(0)}

for label in LABELS:
    groups[label] = train_df[label].eq(1)

for label, mask in groups.items():

    words = word_lengths[mask]
    chars = char_lengths[mask]

    length_groups[label] = words

    length_records.append({
        'Category': label,
        'Comments': len(words),
        'Mean Words': words.mean(),
        'Median Words': words.median(),
        '90th Percentile Words': words.quantile(0.90),
        '95th Percentile Words': words.quantile(0.95),
        '99th Percentile Words': words.quantile(0.99),
        'Maximum Words': words.max(),
        'Mean Characters': chars.mean(),
        'Median Characters': chars.median()
    })

label_length_df = pd.DataFrame(length_records)

display(label_length_df.round(2))

label_length_df.to_csv(
    TABLES_DIR / 'comment_length_by_label.csv',
    index=False
)

# Visualize word length distributions across labels

fig, ax = plt.subplots(figsize=(12, 6))

categories = list(length_groups.keys())

box = ax.boxplot(
    [length_groups[label] for label in categories],
    tick_labels=categories,
    patch_artist=True,
    showfliers=False,
    widths=0.6,
    medianprops={'color': 'black', 'linewidth': 1.8}
)

for patch, label in zip(box['boxes'], categories):
    patch.set_facecolor(LABEL_COLORS.get(label, '#9E9E9E'))
    patch.set_alpha(0.8)

ax.set_title('Comment Word Length Across Toxicity Categories', fontsize=14)
ax.set_xlabel('Comment Category')
ax.set_ylabel('Number of Words')
ax.grid(axis='x', visible=False)

fig.text(
    0.5, 0.01,
    'Outliers are hidden for readability. Statistics use all comments.',
    ha='center',
    fontsize=10
)

sns.despine()
plt.tight_layout(rect=[0, 0.04, 1, 1])

plt.savefig(
    FIGURES_DIR / 'comment_length_by_label.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

In [ ]:
# Analyse surface-level writing patterns in training comments

texts = train_df['comment_text']

letter_count = texts.str.count(r'[A-Za-z]')
uppercase_count = texts.str.count(r'[A-Z]')

uppercase_ratio = uppercase_count / letter_count.replace(0, np.nan)

text_features = pd.DataFrame({
    'High capitalization': (
        (uppercase_ratio >= 0.5) & (letter_count >= 5)
    ),
    'Exclamation marks': texts.str.contains('!', regex=False),
    'Question marks': texts.str.contains('?', regex=False),
    'Repeated punctuation': texts.str.contains(r'[!?]{2,}', regex=True),
    'Repeated letters': texts.str.count(r'([A-Za-z])\1{2,}').gt(0),
    'Second-person pronouns': texts.str.contains(
        r'\b(?:you|your|yours|yourself|yourselves)\b',
        case=False,
        regex=True
    )
})

# Compare writing patterns across comment categories

feature_groups = {
    'Clean': label_cardinality.eq(0),
    'Any toxic': label_cardinality.gt(0)
}

for label in LABELS:
    feature_groups[label] = train_df[label].eq(1)

feature_records = []

for category, mask in feature_groups.items():
    feature_records.append({
        'Category': category,
        'Comments': int(mask.sum()),
        **(text_features.loc[mask].mean() * 100).to_dict()
    })

text_characteristics_df = pd.DataFrame(feature_records)

display(text_characteristics_df.round(2))

text_characteristics_df.to_csv(
    TABLES_DIR / 'text_characteristics.csv',
    index=False
)

# Compare clean and toxic comment characteristics

plot_df = (
    text_characteristics_df
    .set_index('Category')
    .loc[['Clean', 'Any toxic'], text_features.columns]
    .T
)

fig, ax = plt.subplots(figsize=(12, 6))

plot_df.plot(
    kind='barh',
    color=['#4C78A8', '#E45756'],
    width=0.75,
    ax=ax
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        padding=4,
        fontsize=10
    )

ax.set_title('Writing Patterns in Clean and Toxic Comments', fontsize=14)
ax.set_xlabel('Percentage of Comments')
ax.set_ylabel('Text Characteristic')
ax.set_xlim(0, plot_df.to_numpy().max() * 1.25)
ax.legend(title='Comment Category')
ax.grid(axis='y', visible=False)

sns.despine()
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / 'text_characteristics_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

In [ ]:
# Identify duplicate comments, conflicting annotations and dataset overlap

def normalize_text(series):
    return (
        series.str.lower()
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

train_norm = normalize_text(train_df['comment_text'])
test_norm = normalize_text(test_df['comment_text'])
valid_test_norm = normalize_text(test_valid_df['comment_text'])

# Identify exact and normalized training duplicates
exact_counts = train_df['comment_text'].value_counts()
norm_counts = train_norm.value_counts()

exact_groups = exact_counts[exact_counts > 1]
duplicate_groups = norm_counts[norm_counts > 1]

# Identify duplicate groups with conflicting label annotations
duplicate_mask = train_norm.isin(duplicate_groups.index)

duplicate_data = train_df.loc[duplicate_mask, ['id'] + LABELS].copy()
duplicate_data['normalized_text'] = train_norm[duplicate_mask]

conflicting_groups = (
    duplicate_data.groupby('normalized_text')[LABELS]
    .nunique()
    .gt(1)
    .any(axis=1)
)

# Check exact and normalized train-test overlap
exact_test_overlap = test_df['comment_text'].isin(
    train_df['comment_text']
)

full_test_overlap = test_norm.isin(train_norm)
valid_test_overlap = valid_test_norm.isin(train_norm)

train_overlap_mask = train_norm.isin(valid_test_norm)

shared_groups = set(train_norm) & set(valid_test_norm)

# Summarize duplicate and overlap statistics
duplicate_audit_df = pd.DataFrame({
    'Metric': [
        'Exact training duplicate groups',
        'Rows in exact duplicate groups',
        'Normalized training duplicate groups',
        'Rows in normalized duplicate groups',
        'Excess normalized duplicate rows',
        'Normalized groups with conflicting labels',
        'Exact overlapping test comments',
        'Normalized overlapping full test comments',
        'Normalized overlapping labelled test comments',
        'Training rows overlapping labelled test',
        'Shared normalized train-labelled-test groups'
    ],
    'Count': [
        len(exact_groups),
        exact_groups.sum(),
        len(duplicate_groups),
        duplicate_groups.sum(),
        (duplicate_groups - 1).sum(),
        conflicting_groups.sum(),
        exact_test_overlap.sum(),
        full_test_overlap.sum(),
        valid_test_overlap.sum(),
        train_overlap_mask.sum(),
        len(shared_groups)
    ]
})

display(duplicate_audit_df)

# Save audit statistics
duplicate_audit_df.to_csv(
    TABLES_DIR / 'duplicate_overlap_audit.csv',
    index=False
)

# Save overlapping record IDs for data preparation
train_df.loc[train_overlap_mask, ['id']].to_csv(
    TABLES_DIR / 'overlapping_training_ids.csv',
    index=False
)

test_valid_df.loc[valid_test_overlap, ['id']].to_csv(
    TABLES_DIR / 'overlapping_test_ids.csv',
    index=False
)

print("Original training comments:", len(train_df))
print("Training rows overlapping labelled test:", train_overlap_mask.sum())
print("Potential leakage-free development pool:",
      len(train_df) - train_overlap_mask.sum())

print("\nAudit completed. Original datasets remain unchanged.")

In [ ]:
# Analyse identity-term frequencies and toxicity associations

import re

identity_groups = {
    'Religion': [
        'muslim', 'christian', 'hindu', 'jewish',
        'sikh', 'buddhist', 'atheist'
    ],
    'Gender': [
        'man', 'woman', 'male', 'female', 'transgender'
    ],
    'Sexual orientation': [
        'gay', 'lesbian', 'bisexual', 'homosexual', 'straight'
    ],
    'Race and ethnicity': [
        'black', 'white', 'asian', 'african', 'arab'
    ],
    'Nationality and region': [
        'indian', 'pakistani', 'american', 'chinese', 'tamil'
    ],
    'Caste': [
        'dalit', 'brahmin'
    ],
    'Disability': [
        'disabled', 'autistic', 'blind', 'deaf'
    ]
}

any_toxic = train_df[LABELS].sum(axis=1).gt(0)
baseline_rate = any_toxic.mean() * 100

identity_records = []
identity_masks = {}

for group, terms in identity_groups.items():
    for term in terms:

        pattern = r'\b' + re.escape(term) + r'\b'

        mask = train_df['comment_text'].str.contains(
            pattern, case=False, regex=True, na=False
        )

        identity_masks[term] = mask

        total = int(mask.sum())
        toxic = int((mask & any_toxic).sum())
        clean = total - toxic

        rate = toxic / total * 100 if total > 0 else np.nan

        identity_records.append({
            'Identity Group': group,
            'Identity Term': term,
            'Total Comments': total,
            'Clean Comments': clean,
            'Toxic Comments': toxic,
            'Toxicity Rate (%)': rate,
            'Difference from Overall Rate (pp)': rate - baseline_rate
        })

identity_df = pd.DataFrame(identity_records)

display(identity_df.round(2))

identity_df.to_csv(
    TABLES_DIR / 'identity_term_statistics.csv',
    index=False
)

print("Overall toxicity rate:", round(baseline_rate, 2), "%")
print("Identity terms examined:", len(identity_df))

In [ ]:
# Visualize toxicity rates for identity terms with sufficient support

MIN_SUPPORT = 100

plot_df = identity_df[
    identity_df['Total Comments'] >= MIN_SUPPORT
].sort_values('Toxicity Rate (%)')

group_colors = {
    'Religion': '#4C78A8',
    'Gender': '#F58518',
    'Sexual orientation': '#E45756',
    'Race and ethnicity': '#54A24B',
    'Nationality and region': '#B279A2',
    'Caste': '#72B7B2',
    'Disability': '#9D755D'
}

fig, ax = plt.subplots(
    figsize=(11, max(6, len(plot_df) * 0.38))
)

bars = ax.barh(
    plot_df['Identity Term'],
    plot_df['Toxicity Rate (%)'],
    color=plot_df['Identity Group'].map(group_colors)
)

for bar, rate, count in zip(
    bars,
    plot_df['Toxicity Rate (%)'],
    plot_df['Total Comments']
):
    ax.text(
        rate + 0.6,
        bar.get_y() + bar.get_height() / 2,
        f'{rate:.1f}% (n={count:,})',
        va='center',
        fontsize=9
    )

ax.axvline(
    baseline_rate,
    color='black',
    linestyle='--',
    linewidth=1.5,
    label=f'Overall toxicity rate: {baseline_rate:.2f}%'
)

ax.set_title('Toxicity Associations with Identity Terms', fontsize=14)
ax.set_xlabel('Percentage of Comments Annotated as Toxic')
ax.set_ylabel('Identity Term')
ax.set_xlim(0, plot_df['Toxicity Rate (%)'].max() * 1.35)
ax.legend(loc='lower right')
ax.grid(axis='y', visible=False)

fig.text(
    0.5, 0.01,
    'Only terms occurring in at least 100 comments are displayed.',
    ha='center',
    fontsize=10
)

sns.despine()
plt.tight_layout(rect=[0, 0.04, 1, 1])

plt.savefig(
    FIGURES_DIR / 'identity_term_toxicity_associations.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

In [ ]:
# Summarize EDA findings and their methodological implications

eda_findings = [
    {
        'Analysis': 'Class distribution',
        'Finding': '89.83% of comments have no positive toxicity labels.',
        'Methodological implication':
            'Do not rely on accuracy alone; use Macro AP, Macro F1 and per-label metrics.'
    },
    {
        'Analysis': 'Class imbalance',
        'Finding': 'Threat has only 478 positive examples and a 332.83:1 imbalance ratio.',
        'Methodological implication':
            'Compare standard BCE, weighted BCE and focal loss.'
    },
    {
        'Analysis': 'Label cardinality',
        'Finding': '60.80% of toxic comments contain multiple positive labels.',
        'Methodological implication':
            'Use six sigmoid outputs and multilabel stratification.'
    },
    {
        'Analysis': 'Label co-occurrence',
        'Finding': 'Strong overlap exists between several toxicity categories.',
        'Methodological implication':
            'Avoid naive resampling that may distort multiple label distributions.'
    },
    {
        'Analysis': 'Comment length',
        'Finding': 'Median length is 36 words, while the 95th percentile is 230 words.',
        'Methodological implication':
            'Analyse actual tokenizer lengths before selecting maximum sequence length.'
    },
    {
        'Analysis': 'Label-specific lengths',
        'Finding': 'Severe toxic comments have a particularly long upper tail.',
        'Methodological implication':
            'Measure truncation rates separately for every toxicity label.'
    },
    {
        'Analysis': 'Text characteristics',
        'Finding': 'Capitalization and repeated punctuation are more frequent in toxic comments.',
        'Methodological implication':
            'Preserve original text and investigate character-level TF-IDF.'
    },
    {
        'Analysis': 'Duplicate comments',
        'Finding': '235 normalized duplicate groups, including 41 with conflicting labels.',
        'Methodological implication':
            'Keep duplicate groups within one partition and preserve original annotations.'
    },
    {
        'Analysis': 'Train-test overlap',
        'Finding': '32 training records overlap with 40 labelled test records after normalization.',
        'Methodological implication':
            'Remove overlapping training records while retaining the official test set.'
    },
    {
        'Analysis': 'Test label availability',
        'Finding': '63978 fully labelled test comments are available.',
        'Methodological implication':
            'Exclude unlabelled records and preserve the official labelled benchmark.'
    },
    {
        'Analysis': 'Identity-term associations',
        'Finding': 'Some identity terms show substantially higher toxicity annotation rates.',
        'Methodological implication':
            'Conduct natural-comment and controlled counterfactual fairness audits.'
    },
    {
        'Analysis': 'Identity-term representation',
        'Finding': 'Several demographic identity terms have very limited support.',
        'Methodological implication':
            'Use controlled templates alongside natural-comment fairness evaluation.'
    }
]

eda_findings_df = pd.DataFrame(eda_findings)

display(eda_findings_df)

eda_findings_df.to_csv(
    TABLES_DIR / 'eda_findings_and_implications.csv',
    index=False
)

print("Recorded findings:", len(eda_findings_df))

In [ ]:
# Verify saved EDA tables and figures and document the findings

required_tables = [
    'dataset_overview.csv',
    'training_schema.csv',
    'data_quality_summary.csv',
    'label_validity.csv',
    'test_label_availability.csv',
    'empty_test_comments.csv',
    'clean_toxic_distribution.csv',
    'label_statistics.csv',
    'label_cardinality.csv',
    'label_cardinality_summary.csv',
    'label_cooccurrence_counts.csv',
    'label_cooccurrence_percentages.csv',
    'comment_length_statistics.csv',
    'comment_length_by_label.csv',
    'text_characteristics.csv',
    'duplicate_overlap_audit.csv',
    'overlapping_training_ids.csv',
    'overlapping_test_ids.csv',
    'identity_term_statistics.csv',
    'eda_findings_and_implications.csv'
]

required_figures = [
    'clean_toxic_distribution.png',
    'label_distribution.png',
    'class_imbalance_ratio.png',
    'label_cardinality_distribution.png',
    'label_cooccurrence_counts.png',
    'label_cooccurrence_percentages.png',
    'comment_length_distributions.png',
    'comment_length_by_label.png',
    'text_characteristics_comparison.png',
    'identity_term_toxicity_associations.png'
]

verification = []

for category, directory, filenames in [
    ('Table', TABLES_DIR, required_tables),
    ('Figure', FIGURES_DIR, required_figures)
]:
    for filename in filenames:
        path = directory / filename

        verification.append({
            'Type': category,
            'Filename': filename,
            'Status': 'Available' if path.is_file()
            and path.stat().st_size > 0 else 'Missing'
        })

verification_df = pd.DataFrame(verification)

display(verification_df)

assert verification_df['Status'].eq('Available').all(), \
    "Some EDA artifacts are missing or empty"

verification_df.to_csv(
    TABLES_DIR / 'eda_artifact_verification.csv',
    index=False
)

# Save EDA summary

summary = """# Notebook 01 - Exploratory Data Analysis

## Dataset Overview
- Original training comments: 159,571
- Official test comments: 153,164
- Fully labelled test comments: 63,978
- Unlabelled test comments: 89,186
- Target labels: 6

## Major Findings
- 89.83% of training comments have no positive labels.
- Threat is the rarest category with 478 positive examples.
- The threat imbalance ratio is approximately 332.83:1.
- 60.80% of toxic comments contain multiple positive labels.
- Strong co-occurrence exists between toxicity categories.
- Median comment length is 36 words.
- The 95th percentile comment length is 230 words.
- Severe toxic comments exhibit a particularly long upper tail.
- Capitalization and repeated punctuation are more frequent
  in toxic comments.
- 235 normalized duplicate groups were identified.
- 41 duplicate groups contain conflicting annotations.
- 32 training records overlap with labelled test comments.
- Identity-term associations motivate a model-level fairness audit.

## Methodological Implications
- Use group-aware multilabel stratification.
- Remove detected training-side overlaps with the labelled test set.
- Preserve original annotations and raw comment text.
- Investigate representative subsampling and imbalance-aware losses.
- Analyse actual tokenizer lengths before selecting sequence length.
- Use Macro AP, Macro F1 and per-label evaluation metrics.
- Optimize classification thresholds using validation data.
- Conduct natural-comment and counterfactual fairness audits.

## Conclusion
Exploratory data analysis is complete. Original datasets remain
unchanged. No model training or permanent data splitting was performed.

## Next Stage
Notebook 02 - Data Preparation.
"""

(RESULTS_DIR / 'summary.md').write_text(
    summary, encoding='utf-8'
)

print("\nEDA completion summary")
print("Analysis tables verified:", len(required_tables))
print("Figures verified:", len(required_figures))
print("Findings documented:", len(eda_findings_df))
print("Missing artifacts:", verification_df['Status'].eq('Missing').sum())
print("EDA completed successfully.")